# Assignment 2 Diagnose and improve a generator
**VAE track · Notebook v5 · Chonnam National University · Fall 2026**

Choose **VAE OR GAN**, not both. 두 트랙 중 **하나만** 선택하세요.
Released 15 October 2026 · Due 4 November 2026, 23:59 KST · 12 points · 10% of course grade.

## Your task / 과제의 목표
Inspect a baseline generator, identify a visible limitation, change **one** setting, and decide whether the generated distribution improved. Improvement is not guaranteed and is not required for full marks.

Baseline의 실제 생성 결과에서 문제를 찾고, 설정 하나를 바꾼 뒤 개선 여부를 판단합니다. **좋아진 결과 자체가 아니라 근거 있는 판단을 평가합니다.**

This is the Week 4 VAE architecture on MNIST. The baseline deliberately uses a large KL weight, beta = 20. Change only beta. / 4주차 VAE 구조를 사용합니다. Baseline에는 교육용으로 큰 KL weight인 beta = 20을 적용했습니다. beta만 바꿉니다.

**Active work: about 3–4 hours, up to 5 if debugging is needed.** CPU is sufficient. Only two training runs are required. Expect a few minutes for the pair on Colab; actual time depends on the machine and data download.

직접 작업 시간은 약 3–4시간이며, 오류 해결이 필요하면 5시간까지 예상합니다. Baseline 1회와 변경 실험 1회만 수행합니다. 추가 seed 실험이나 hyperparameter 탐색은 요구하지 않습니다.

1. Save a copy in Drive and run Setup. / Drive에 사본을 저장하고 Setup을 실행합니다.
2. Run the baseline and inspect generated samples. / Baseline을 실행하고 생성 결과를 관찰합니다.
3. Write your diagnosis and commit ONE change before seeing its result. / 변경 실험을 보기 전에 관찰·설정·예측·판단 기준을 작성합니다.
4. Complete three short code blanks or use the provided code, then train the intervention. / 코드 세 칸을 완성하거나 제공 코드로 전환한 뒤 변경 실험을 실행합니다.
5. Compare samples and metrics, export the receipt, and write a short report. / 생성 결과와 지표를 비교하고 결과 기록 PDF와 짧은 보고서를 작성합니다.

Submit the executed notebook and ONE PDF containing your 1–2 page analysis followed by the two-page receipt. See `A2_Student_Assignment_v6.pdf` for the rubric. English or Korean answers are accepted.

실행한 노트북과 PDF 하나를 제출하세요. PDF는 분석 본문 1–2쪽 뒤에 자동 생성한 결과 기록 2쪽을 붙입니다. 영어·한국어 답안 모두 가능합니다.

## 0 Setup / 준비
Run these supplied cells without editing. No account, Drive mount, GPU, or paid service is needed.

준비 셀은 수정하지 않고 실행하세요. 계정 연결이나 Drive mount가 필요하지 않습니다. 코드의 제공 함수는 공개된 실행 지원 도구이며, 숨겨진 정답이 아닙니다. Reference 사용은 감점하지 않고 결과 기록에 표시합니다. 분석 답안은 직접 작성해야 합니다.

In [ ]:
#@title Setup and workflow helpers / 준비와 실행 관리 {display-mode:"form"}
import copy, gzip, hashlib, json, math, os, platform, struct, time, urllib.request
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display, FileLink

torch.set_num_threads(2)
VERSION = 'A2 redesign v6 / notebook v5'
SEED = 0
BASE, VARIANT, PLAN, AUDIT = None, None, None, None
CODE_STATUS = {}
OUT = Path('a2_outputs'); OUT.mkdir(exist_ok=True)

def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()[:16]

def array_id(a):
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()[:16]

def code_id():
    # Detect ordinary edits after a run; this is not an anti-tampering system.
    return fingerprint({name: (globals()[name].__code__.co_code.hex(), repr(globals()[name].__code__.co_consts)) for name in CODE_NAMES})

def check_code(number, use_reference=False):
    name = CODE_NAMES[number - 1]
    if use_reference:
        globals()[name] = REFERENCES[number - 1]
    try:
        for args in TESTS[number - 1]:
            expected = REFERENCES[number - 1](*copy.deepcopy(args))
            actual = globals()[name](*copy.deepcopy(args))
            if actual is None:
                raise ValueError('빈칸이 남아 있습니다 / the code blank is still None')
            a, b = torch.as_tensor(actual), torch.as_tensor(expected)
            assert a.shape == b.shape and torch.allclose(a.float(), b.float(), atol=1e-6, rtol=1e-5)
            if b.requires_grad:
                assert a.requires_grad, 'gradient 연결을 유지하세요 / preserve autograd'
        mode = 'reference' if globals()[name] is REFERENCES[number - 1] else 'student'
        CODE_STATUS[name] = mode
        print(f'Code {number}: PASS ({mode})')
        return True
    except Exception as exc:
        CODE_STATUS[name] = 'pending'
        print(f'Code {number}: NOT READY. {type(exc).__name__}: {exc}')
        print('식과 tensor shape을 확인하세요. 필요하면 check_code(number, use_reference=True)를 사용하세요. 감점은 없습니다.')
        return False

def record_plan(student_id, diagnosis, candidate, prediction, success_rule):
    global PLAN, VARIANT, AUDIT
    if BASE is None:
        print('먼저 baseline을 실행하세요 / Run the baseline first.'); return
    values = [student_id, diagnosis, prediction, success_rule]
    if any(not isinstance(x, str) or not x.strip() for x in values):
        print('NOT READY: 학번, baseline 관찰, 예측, 판단 기준을 작성하세요.'); return
    if candidate not in OPTIONS:
        print('NOT READY: 제공된 두 설정 중 하나를 선택하세요.'); return
    draft = dict(student_id=student_id.strip(), diagnosis=diagnosis.strip(), candidate=float(candidate),
                 prediction=prediction.strip(), success_rule=success_rule.strip(), baseline_id=BASE['id'])
    if PLAN != draft:
        PLAN = draft; VARIANT = None; AUDIT = None
    print('PLAN SAVED. 변경 실험을 실행하세요 / Run your chosen intervention.')

def run_baseline():
    global BASE, VARIANT, PLAN, AUDIT
    cfg = config(BASE_VALUE)
    if BASE is not None and BASE['config'] == cfg:
        print('기존 baseline 재사용 / Reusing baseline.'); return
    BASE = train(BASE_VALUE, student=False)
    PLAN = VARIANT = AUDIT = None

def run_variant():
    global VARIANT, AUDIT
    if PLAN is None or BASE is None:
        print('NOT READY: baseline을 보고 변경 계획을 먼저 작성하세요.'); return
    if BASE['config'] != config(BASE_VALUE) or PLAN['baseline_id'] != BASE['id']:
        print('NOT READY: 설정이 바뀌었습니다. baseline부터 다시 실행하세요.'); return
    if not all([check_code(i) for i in [1, 2, 3]]):
        print('NOT READY: 코드 세 칸을 완성하거나 제공 코드로 전환하세요.'); return
    key = fingerprint(PLAN)
    if VARIANT is not None and VARIANT['plan_id'] == key and VARIANT['code_id'] == code_id():
        print('기존 변경 실험 재사용 / Reusing intervention run.'); return
    VARIANT = train(PLAN['candidate'], student=True)
    VARIANT.update(plan_id=key, code_id=code_id())
    AUDIT = None

def ready_reason():
    if BASE is None or PLAN is None or VARIANT is None:
        return 'baseline, plan 또는 intervention 실행이 남아 있습니다.'
    if BASE['config'] != config(BASE_VALUE) or VARIANT['config'] != config(PLAN['candidate']):
        return '설정이 바뀌었습니다. baseline부터 다시 실행하세요.'
    if VARIANT['plan_id'] != fingerprint(PLAN) or PLAN['baseline_id'] != BASE['id']:
        return '변경 계획과 실행 결과가 다릅니다.'
    if VARIANT['code_id'] != code_id() or any(CODE_STATUS.get(n) == 'pending' for n in CODE_NAMES):
        return '코드가 바뀌었습니다. 코드 점검과 변경 실험을 다시 실행하세요.'
    return ''

def make_receipt(ai_use):
    reason = ready_reason()
    if not reason and TRACK == 'VAE' and 'OFFLINE' in DATA_NAME:
        reason = '오프라인 smoke test입니다. 제출 전 MNIST로 두 run을 실행하세요.'
    if not reason and (not isinstance(ai_use, str) or not ai_use.strip()):
        reason = 'AI 사용 내용을 쓰세요. 사용하지 않았다면 none을 적으세요.'
    if not reason and TRACK == 'VAE' and (AUDIT is None or AUDIT['run_id'] != VARIANT['id']):
        reason = '두 grid의 sample 01–16에 대한 visual audit를 작성하세요.'
    if reason:
        print('NOT READY:', reason); return None
    b, v = metrics(BASE), metrics(VARIANT)
    payload = dict(version=VERSION, track=TRACK, plan=PLAN, code=CODE_STATUS.copy(),
                   baseline=dict(config=BASE['config'], metrics=b, sample_id=array_id(BASE['samples'])),
                   intervention=dict(config=VARIANT['config'], metrics=v, sample_id=array_id(VARIANT['samples'])),
                   visual_audit=AUDIT if TRACK == 'VAE' else None, ai_use=ai_use.strip(),
                   environment=dict(python=platform.python_version(), torch=torch.__version__, numpy=np.__version__, device='cpu'))
    fid=fingerprint(payload)
    receipt=dict(status='READY', fingerprint=fid, **payload)
    (OUT/f'A2_{TRACK}_receipt.json').write_text(json.dumps(receipt,ensure_ascii=False,indent=2),encoding='utf8')
    # English labels and numbers in the PDF avoid missing Korean fonts on Colab.
    # The full original Korean plan/AI statement is preserved in the notebook and JSON.
    lines=[f'{VERSION} | {TRACK} | READY',f'Fingerprint: {fid}',
           f'Data: {BASE["config"]["data"]}', f'Changed parameter: {PARAMETER}',
           f'Baseline: {BASE_VALUE}   Intervention: {PLAN["candidate"]}',
           'Only the named parameter changes; both runs restart from the same seed.',
           'Settings: '+json.dumps(BASE['config'],ensure_ascii=True), '',
           'BASELINE', *[f'  {k}: {v}' for k,v in b.items()], '',
           'INTERVENTION', *[f'  {k}: {v}' for k,v in v.items()], '',
           'Code: '+str(CODE_STATUS),
           'Full plan, AI statement and annotations: notebook and receipt JSON.',
           'This receipt is an appendix. Write your own analysis in the report.',
           'Fingerprint checks consistency; it does not prove authorship.']
    import textwrap
    wrapped='\n'.join(y for x in lines for y in (textwrap.wrap(x,95) or ['']))
    pdf_path=OUT/f'A2_{TRACK}_receipt.pdf'
    with PdfPages(pdf_path) as pdf:
        fig=plt.figure(figsize=(8.27,11.69));fig.text(.07,.95,wrapped,va='top',fontfamily='monospace',fontsize=8.5);pdf.savefig(fig);plt.close(fig)
        fig=comparison_figure();pdf.savefig(fig,bbox_inches='tight');plt.close(fig)
    print('READY | fingerprint:',fid)
    print('Baseline:',b); print('Intervention:',v)
    print('분석 보고서 뒤에 PDF를 붙이세요. JSON은 노트북과 함께 보관하면 됩니다.')
    display(FileLink(str(pdf_path)));display(FileLink(str(OUT/f'A2_{TRACK}_receipt.json')))
    return receipt

In [ ]:
#@title VAE model and supplied training loop / model과 학습 코드 {display-mode:"form"}
TRACK='VAE'
PARAMETER='KL weight beta'
BASE_VALUE=20.0
OPTIONS=(1.0,0.1)
EPOCHS=6; BATCH=256; LATENT=2; LR=0.001
DATA_SOURCE='auto'  # auto: MNIST; offline: bundled sklearn digits, smoke test only
DATA_NAME=None
Z_FIXED=torch.randn(64,LATENT,generator=torch.Generator().manual_seed(2026))

class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=nn.Sequential(nn.Linear(784,512),nn.ReLU(),nn.Linear(512,256),nn.ReLU())
        self.head_mu=nn.Linear(256,LATENT);self.head_logvar=nn.Linear(256,LATENT)
        self.dec=nn.Sequential(nn.Linear(LATENT,256),nn.ReLU(),nn.Linear(256,512),nn.ReLU(),nn.Linear(512,784))
    def encode(self,x):
        h=self.enc(x);return self.head_mu(h),self.head_logvar(h)
    def decode(self,z):return self.dec(z)

def reference_sample_z(mu,logvar,eps):
    return mu+torch.exp(0.5*logvar)*eps
def reference_vae_objective(rec,kl,beta):
    return rec+beta*kl
def reference_readable_share(labels):
    return float(np.mean(np.asarray(labels)!='x'))
CODE_NAMES=['sample_z','vae_objective','readable_share']
REFERENCES=[reference_sample_z,reference_vae_objective,reference_readable_share]
TESTS=[[(torch.tensor([[0.,1.]],requires_grad=True),torch.tensor([[0.,math.log(4.)]]),torch.tensor([[1.,-1.]])),
        (torch.zeros(2,2),torch.zeros(2,2),torch.zeros(2,2))],
       [(torch.tensor(2.,requires_grad=True),torch.tensor(3.),0.1),(torch.tensor(2.),torch.tensor(3.),20.)],
       [(['x','1','2','x'],),(['x','x'],),(['0','1'],)]]

def load_data(source='auto'):
    global X_TRAIN,X_TEST,DATA_NAME,DATA_SOURCE
    DATA_SOURCE=source
    if source not in ['auto','offline']:raise ValueError('source must be auto or offline')
    try:
        if source=='offline':raise OSError('offline mode requested')
        arrays=[]
        for name in ['train-images-idx3-ubyte.gz','t10k-images-idx3-ubyte.gz']:
            p=Path(name)
            if not p.exists():
                last=None
                for base in ['https://ossci-datasets.s3.amazonaws.com/mnist/','https://storage.googleapis.com/cvdf-datasets/mnist/']:
                    try:
                        with urllib.request.urlopen(base+name,timeout=8) as response:raw=response.read()
                        # Validate before caching; a failed download must not become a valid cache.
                        decoded=gzip.decompress(raw);magic,n,r,c=struct.unpack('>IIII',decoded[:16])
                        assert magic==2051 and r==28 and c==28 and len(decoded)==16+n*r*c
                        p.write_bytes(raw);break
                    except Exception as exc:last=exc
                else:raise OSError(f'MNIST download failed: {last}')
            with gzip.open(p,'rb') as f:
                magic,n,r,c=struct.unpack('>IIII',f.read(16));assert magic==2051 and r*c==784
                arrays.append(torch.from_numpy(np.frombuffer(f.read(),np.uint8).copy().reshape(n,784)).float()/255)
        X_TRAIN,X_TEST=arrays;DATA_NAME='MNIST 60000 train / 10000 test'
    except (OSError,AssertionError,ValueError,EOFError,struct.error) as exc:
        from sklearn.datasets import load_digits
        from sklearn.model_selection import train_test_split
        digits=load_digits();x=torch.tensor(digits.images/16,dtype=torch.float32)[:,None]
        x=F.interpolate(x,size=(28,28),mode='bilinear',align_corners=False).flatten(1)
        tr,te=train_test_split(np.arange(len(x)),test_size=.2,random_state=42,stratify=digits.target)
        X_TRAIN,X_TEST=x[tr],x[te];DATA_NAME='OFFLINE sklearn digits SMOKE TEST ONLY'
        print('MNIST unavailable:',type(exc).__name__)
        print('오프라인 실행 점검용 data입니다. MNIST 결과로 해석하거나 제출하지 마세요.')
    print(DATA_NAME,tuple(X_TRAIN.shape),tuple(X_TEST.shape))

def config(value):
    return dict(data=DATA_NAME,seed=SEED,epochs=EPOCHS,batch=BATCH,latent=LATENT,lr=LR,beta=float(value),
                train_size=len(X_TRAIN),test_size=len(X_TEST),optimizer='Adam default betas',eval_noise_seed=2026)

def train(value,student=False):
    start=time.time();torch.manual_seed(SEED);model=VAE();opt=torch.optim.Adam(model.parameters(),lr=LR)
    history=[]
    for ep in range(1,EPOCHS+1):
        perm=torch.randperm(len(X_TRAIN));rt=kt=0.
        for i in range(0,len(X_TRAIN),BATCH):
            x=X_TRAIN[perm[i:i+BATCH]];mu,lv=model.encode(x);eps=torch.randn_like(mu)
            z=(sample_z if student else reference_sample_z)(mu,lv,eps)
            rec=F.binary_cross_entropy_with_logits(model.decode(z),x,reduction='sum')/len(x)
            # Match the Week 4 lab implementation of KL(q || p).
            q=torch.distributions.Normal(mu,torch.exp(.5*lv));p=torch.distributions.Normal(torch.zeros_like(mu),torch.ones_like(mu))
            kl=torch.distributions.kl_divergence(q,p).sum(1).mean()
            loss=(vae_objective if student else reference_vae_objective)(rec,kl,value)
            opt.zero_grad();loss.backward();opt.step();rt+=float(rec.detach())*len(x);kt+=float(kl.detach())*len(x)
        history.append([ep,rt/len(X_TRAIN),kt/len(X_TRAIN)])
        print(f'epoch {ep}/{EPOCHS} | reconstruction BCE {history[-1][1]:.2f} | KL {history[-1][2]:.2f}')
    model.eval()
    with torch.no_grad():
        mu,lv=model.encode(X_TEST)
        eps=torch.randn(mu.shape,generator=torch.Generator().manual_seed(12345))
        z=reference_sample_z(mu,lv,eps)
        rec=float(F.binary_cross_entropy_with_logits(model.decode(z),X_TEST,reduction='sum'))/len(X_TEST)
        kl=float((.5*(mu.square()+lv.exp()-1-lv)).sum(1).mean())
        samples=torch.sigmoid(model.decode(Z_FIXED)).numpy()
        recon=torch.sigmoid(model.decode(mu[:8])).numpy()
    assert np.isfinite(samples).all()
    result=dict(config=config(value),samples=samples,recon=recon,history=history,rec=rec,kl=kl,seconds=round(time.time()-start,2))
    result['id']=fingerprint([result['config'],array_id(samples)])
    print('Completed in',result['seconds'],'s. beta =',value)
    return result

def image_grid(fig,slot,images,title,number=True,rows=4):
    sub=slot.subgridspec(rows,8,wspace=.03,hspace=.3)
    for i in range(min(rows*8,len(images))):
        ax=fig.add_subplot(sub[i//8,i%8]);ax.imshow(images[i].reshape(28,28),cmap='gray',vmin=0,vmax=1);ax.axis('off')
        if number:ax.set_title(f'{i+1:02d}',fontsize=7,pad=1)
    # Group title is drawn by the caller, outside the image axes.

def baseline_figure():
    if BASE is None:return
    fig=plt.figure(figsize=(10,6));gs=fig.add_gridspec(1,2)
    image_grid(fig,gs[0],X_TEST[:32].numpy(),'Real test images');image_grid(fig,gs[1],BASE['samples'][:32],'Baseline')
    fig.text(.26,.97,'Real test images',ha='center',fontsize=12);fig.text(.75,.97,'Baseline | prior generations',ha='center',fontsize=12)
    plt.show();plt.close(fig)

def comparison_figure():
    fig=plt.figure(figsize=(11,10));gs=fig.add_gridspec(2,2,height_ratios=[4,3],hspace=.38)
    image_grid(fig,gs[0,0],BASE['samples'][:32],'Baseline');image_grid(fig,gs[0,1],VARIANT['samples'][:32],'Intervention')
    fig.text(.30,.94,f'Baseline beta={BASE_VALUE} | prior samples',ha='center',fontsize=11)
    fig.text(.73,.94,f'Intervention beta={PLAN["candidate"]} | same z',ha='center',fontsize=11)
    lower=gs[1,:].subgridspec(3,8,wspace=.03,hspace=.15)
    for row,images in enumerate([X_TEST[:8].numpy(),BASE['recon'],VARIANT['recon']]):
        for i,im in enumerate(images):
            a=fig.add_subplot(lower[row,i]);a.imshow(im.reshape(28,28),cmap='gray',vmin=0,vmax=1);a.set_xticks([]);a.set_yticks([])
            for spine in a.spines.values():spine.set_visible(False)
            if i==0:
                a.set_ylabel(['Original','Baseline','Intervention'][row],fontsize=9,rotation=0,ha='right',va='center')
                a.yaxis.set_label_coords(-.12,.5)
    fig.suptitle('Generation first | reconstruction is a separate task',fontsize=14,y=.99)
    return fig

def save_audit(baseline_labels,intervention_labels):
    global AUDIT
    if ready_reason():print('NOT READY: 먼저 변경 실험을 완료하세요.');return
    parsed=[]
    for text in [baseline_labels,intervention_labels]:
        labels=[x.strip().lower() for x in text.split(',')]
        if len(labels)!=16 or any(x not in list('0123456789')+['x'] for x in labels):
            AUDIT=None;print('NOT READY: 각 grid의 01–16을 순서대로 0–9 또는 x로 쓰세요. 쉼표로 구분합니다.');return
        parsed.append(labels)
    AUDIT=dict(run_id=VARIANT['id'],baseline=parsed[0],intervention=parsed[1])
    print('Visual audit saved. 이 결과는 16개 sample에 대한 주관적 점검이며 전체 생성 성능 점수가 아닙니다.')

def metrics(run):
    out=dict(test_reconstruction_BCE=round(run['rec'],3),test_KL_nats=round(run['kl'],3))
    if AUDIT is not None and VARIANT is not None and AUDIT['run_id']==VARIANT['id']:
        labels=AUDIT['baseline' if run is BASE else 'intervention']
        out.update(audited_samples=16,readable_share=round(float(readable_share(labels)),4),
                   distinct_readable_digits=len(set(labels)-{'x'}),labels=labels)
    return out

def show_comparison():
    reason=ready_reason()
    if reason:print('NOT READY:',reason);return
    fig=comparison_figure();fig.savefig(OUT/'VAE_comparison.png',dpi=140);plt.show();plt.close(fig)
    print('Baseline',metrics(BASE));print('Intervention',metrics(VARIANT))
    print('Grid 01–16을 같은 기준으로 판정하세요. 애매한 이미지는 x입니다. 번호를 바꾸거나 sample을 다시 뽑지 마세요.')

def optional_loss_plot():
    if ready_reason():return
    fig,axes=plt.subplots(1,2,figsize=(9,3))
    for run,label in [(BASE,'Baseline'),(VARIANT,'Intervention')]:
        h=np.array(run['history']);axes[0].plot(h[:,0],h[:,1],label=label);axes[1].plot(h[:,0],h[:,2],label=label)
    for a,title in zip(axes,['Training reconstruction BCE','Training KL (nats per image)']):a.set(xlabel='Epoch',title=title);a.legend()
    fig.tight_layout();plt.show();plt.close(fig)

In [ ]:
#@title Data / MNIST data
load_data('auto')
# If internet is unavailable, the loader uses sklearn digits only to test execution.
# 오프라인 fallback은 실행 점검용입니다. 제출용 MNIST와 다른 data이며 READY PDF는 만들지 않습니다.
# To test it explicitly, use load_data('offline'); restore 'auto' before submitting.

## 1 Observe the baseline / Baseline 관찰
Run the baseline once. Compare real examples with generated samples before reading any loss curve. Keep the original output even if it differs from another student's output.

Baseline을 먼저 실행하세요. Loss curve보다 실제 생성 결과를 먼저 봅니다. 다른 학생과 결과가 달라도 자신의 결과를 그대로 분석하세요.

The baseline uses beta = 20 as an intentionally strong-regularization case, not as a recommended VAE default. Generated images come from z drawn from N(0, I), without an input image. Displayed grayscale images are sigmoid decoder outputs, not sampled binary pixels.

Baseline은 큰 KL weight의 영향을 관찰하는 교육용 조건이며 권장 기본값이 아닙니다. 생성 이미지는 입력 이미지 없이 prior z ~ N(0, I)에서 만듭니다. 화면에는 decoder의 sigmoid 출력을 회색조로 표시합니다. 서로 비슷한 형태만 반복되는지, digit을 구별할 수 있는지 관찰하세요.

In [ ]:
#@title Train baseline and inspect samples / Baseline 실행
run_baseline()
baseline_figure()

## 2 Commit one change / 변경 계획
Choose 1.0 or 0.1. Only KL weight beta changes. Both runs restart from the same initial weights, seed, training length, data order, and training noise. The intervention is NOT a continuation of the baseline. Evaluation uses the same z vectors.

KL weight beta 하나만 변경합니다. 두 run은 같은 초기 weight에서 각각 처음부터 학습하며, data 순서와 training noise 및 평가용 z도 같습니다. Baseline을 이어서 학습하지 않습니다.

Beta controls the relative weight of KL in reconstruction BCE + beta × KL. Lower beta gives reconstruction more relative influence; too little KL pressure can weaken matching to the sampling prior. Smaller beta is not guaranteed to give better prior generations.

Beta는 reconstruction BCE + beta × KL에서 KL의 상대적 비중을 정합니다. 값을 낮추면 reconstruction의 상대적 영향이 커지지만, 너무 작으면 prior와의 일치가 약해질 수 있습니다. 생성 성능을 직접 확인해야 합니다.

Write one or two sentences per field. Name an expected benefit AND a possible cost in your success rule. Commit the plan before running the intervention; do not rewrite the prediction after seeing its result.

각 칸에 한두 문장으로 작성하세요. 판단 기준에는 기대하는 개선과 함께 확인할 손해를 적으세요. 변경 실험 결과를 본 뒤 원래 예측을 고치지 말고, 보고서에서 차이를 설명하세요.

In [ ]:
#@title Record the plan before the intervention / 변경 전 계획
student_id = '' #@param {type:"string"}
diagnosis = '' #@param {type:"string"}
new_value = 1.0 #@param [1.0, 0.1] {type:"raw"}
prediction = '' #@param {type:"string"}
success_rule = '' #@param {type:"string"}
record_plan(student_id, diagnosis, new_value, prediction, success_rule)

## 3 Three short code blanks / 코드 세 칸
Complete one expression in each cell. The self-check uses small synthetic inputs, so it does not reveal the experiment outcome. Keep tensor gradients connected where needed.

각 셀에서 식 하나만 채우세요. 작은 입력으로 계산을 점검하며, 실험 결과의 정답을 알려 주는 검사는 아닙니다. 필요한 곳에서는 autograd 연결을 유지하세요.

If stuck, change `check_code(1)` to `check_code(1, use_reference=True)` (use 2 or 3 for the other cells). There is no penalty; explain what the code computes in your PDF. If you introduce a SyntaxError, restore the answer line to `None` before switching.

해결되지 않으면 해당 셀을 제공 코드로 전환하세요. 감점은 없지만 PDF의 코드 설명은 작성해야 합니다. SyntaxError가 있으면 답안 줄을 먼저 `None`으로 되돌리세요.

### Code 1 Reparameterization
Return mu + standard deviation × eps. logvar is log(variance), not log(standard deviation). eps is supplied so both runs use the same noise. / log variance로부터 standard deviation을 계산하고 제공된 eps를 사용하세요.

In [ ]:
def sample_z(mu, logvar, eps):
    answer = None  # TODO: one expression / 식 하나
    return answer

check_code(1)

### Code 2 Weighted VAE objective
rec and kl are batch means per image. Return reconstruction BCE + beta × KL. / 이미지당 batch 평균인 두 항을 beta에 맞게 결합하세요.

In [ ]:
def vae_objective(rec, kl, beta):
    answer = None  # TODO: one expression / 식 하나
    return answer

check_code(2)

### Code 3 Readable share in the visual audit
labels contains digit strings and x for unclear samples. Return the fraction that is not x. This is a subjective audit, not a classifier accuracy. / x가 아닌 label의 비율을 계산하세요. classifier accuracy가 아닙니다.

In [ ]:
def readable_share(labels):
    answer = None  # TODO: one expression / 식 하나
    return answer

check_code(3)

In [ ]:
#@title Train your ONE intervention / 변경 실험 한 번 실행
run_variant()

## 4 Judge the generated results / 생성 결과 평가
Use the sample comparison first. Then use the table to support or revise your visual impression. You can conclude improved, mixed, unchanged, or worse. Do not run both options to choose the prettier result.

그림에서 먼저 관찰하고 수치로 확인하세요. 개선·장단점이 섞임·차이 없음·악화 중 어느 결론도 가능합니다. 두 선택지를 모두 실행해 보기 좋은 결과만 고르지 마세요.

The top panels show the first 32 of 64 fixed prior generations; the same z is used in both runs. There is no one-to-one real target for these images. The bottom panels show the same eight test originals and reconstruction from encoder means. Reconstruction has an input image; prior generation does not.

위에는 고정된 prior sample 64개 중 앞의 32개를 표시합니다. 두 run의 z는 같지만 latent 의미가 같다는 보장은 없습니다. 생성 이미지에는 대응되는 정답 원본이 없습니다. 아래 reconstruction은 동일한 test image 8장을 입력한 encoder mean으로 만듭니다.

Test reconstruction BCE and KL support the explanation; neither is a direct generated-image quality score. BCE is the lab's grayscale reconstruction cost, not an exact likelihood of grayscale images. KL is measured in nats per image. No total-loss ranking across beta values is used.

Test reconstruction BCE와 KL은 설명을 보조합니다. 생성 이미지 quality를 직접 측정하지 않으며, beta가 다른 전체 loss 값의 크기로 순위를 정하지 않습니다.

In [ ]:
#@title Generated samples and supporting measurements / 생성 결과와 지표
show_comparison()

### Small visual audit / 작은 visual audit
Inspect samples **01–16** in each prior grid, left to right and top to bottom. Record a single digit 0–9 only if its shape is clear enough that you would not need to guess; otherwise use `x`. Use the SAME standard for both runs. Do not relabel an ambiguous image to make the result look better.

각 prior grid의 **01–16**을 순서대로 확인하세요. 추측하지 않고 하나의 digit으로 읽을 수 있을 때만 0–9를 적고, 애매하면 `x`를 적으세요. 두 run에 같은 기준을 적용합니다. 이 입력에는 약 5–10분이 걸립니다.

The notebook reports readable share and the number of distinct readable digits. These describe only this small, subjective audit. They do not estimate full distribution coverage reliably, and repeated labels do not by themselves prove mode collapse. Cite two sample IDs in your report to make your judgment reviewable.

Readable share와 서로 다른 digit 수는 이 16개에 대한 주관적 점검입니다. 전체 distribution coverage를 정확하게 추정하지 못합니다. 보고서에 sample 번호 두 개를 인용해 판단 근거를 보여 주세요.

In [ ]:
#@title Label prior samples 01–16 / 쉼표로 구분해 입력
baseline_labels = '' #@param {type:"string"}
intervention_labels = '' #@param {type:"string"}
save_audit(baseline_labels, intervention_labels)

### Optional training curves / 선택 확인
Use these only if they help explain an observation. You do not need to explain every bend, derive logit gradients, or interpret reference loss constants. Curves alone do not decide generation quality.

관찰한 결과를 설명하는 데 필요할 때만 확인하세요. 모든 굴곡을 설명하거나 logit gradient를 유도할 필요는 없습니다.

In [ ]:
#@title Optional curves / 필요한 경우에만 확인
show_optional_curves = False #@param {type:"boolean"}
if show_optional_curves:
    optional_loss_plot()

## 5 Your report / 분석 보고서
Write about 1–2 pages. The saved plan remains in the notebook; summarize it rather than copying all fields or tables. Include three short code explanations, one sentence per blank.

본문은 1–2쪽 정도로 작성하세요. 계획과 표를 전부 옮겨 적지 말고 필요한 내용만 인용합니다. 코드 칸별 계산 내용과 사용 목적도 한 문장씩 설명하세요.

**Q1 Baseline diagnosis (2 points).** Identify one concrete generation problem and support it with two observations. Distinguish prior generation from reconstruction; cite sample IDs. / 생성 결과의 문제 하나를 정하고 sample 번호를 근거로 두 가지 관찰을 제시하세요. Prior generation과 reconstruction을 구분하세요.

**Q2 One controlled change (2 points).** Explain your choice and original prediction. Name what stayed fixed and why the comparison is fair. State one possible cost you planned to check. / 설정을 선택한 이유와 원래 예측을 설명하세요. 고정한 조건과 그 이유, 함께 확인할 손해를 적으세요.

**Q3 Did generation improve (4 points).** Give a verdict using one direct visual comparison, two relevant numeric comparisons, and one limitation or remaining problem. Distinguish observations from proposed explanations. Explain whether your prediction held. A well-supported negative result earns full credit. / 그림 비교 하나와 수치 비교 두 가지를 근거로 결론을 쓰세요. 남은 문제 또는 판단의 한계 하나를 제시하고, 관찰 사실과 원인 추정을 구분하세요. 예측과 달라도 타당한 분석이면 만점을 받을 수 있습니다.

**Q4 Next experiment and project bridge (2 points).** Propose ONE follow-up that tests an unresolved explanation, with a result that would change your conclusion. Then name a possible final-project dataset/task and a matching sample-quality or coverage check. No extra training or final-project commitment is required. / 아직 확인하지 못한 설명을 검증할 다음 실험 하나와 결론을 바꿀 결과를 적으세요. 이어서 기말 프로젝트 후보 data 또는 task 하나와 그에 맞는 생성 결과 점검 방법을 제시하세요. 추가 실행이나 주제 확정은 요구하지 않습니다.

**Code and reproducibility (2 points).** Three blanks: 0.25 execution + 0.25 correct explanation each; 0.5 for a READY receipt matching the submitted notebook. Reference code has no penalty. A numeric typo is penalized in the affected analysis item only, not again in reproducibility.

코드 세 칸은 각각 실행 0.25점과 설명 0.25점입니다. READY 결과 기록과 제출 노트북이 일치하면 0.5점입니다. 제공 코드 사용은 감점하지 않습니다. 숫자 오타를 두 문항에서 중복 감점하지 않습니다.

## 6 Export the analysis receipt / 결과 기록 저장
Write your AI-use statement (`none` if unused), then run this cell. It produces a two-page PDF appendix and a JSON record in `a2_outputs`. The PDF contains settings, metrics, a fingerprint, and the sample comparison. Your original plan and AI statement remain in the notebook and JSON, including Korean text.

AI 사용 내용과 용도를 적으세요. 사용하지 않았으면 `none`이라고 적습니다. 생성된 결과 기록 PDF 2쪽을 분석 본문 뒤에 붙여 PDF 하나로 제출하세요. JSON은 보관용이며 별도 제출할 필요가 없습니다. Colab의 왼쪽 Files에서 파일을 다운로드할 수 있습니다.

Before submission, use Runtime > Run all once; cached runs are reused unless settings or code changed. Review the regenerated receipt. Save the notebook with outputs. If restarting the runtime, both runs execute again from their seeds. If results change, update the analysis using the new observations while keeping the original prediction.

제출 전에 Runtime > Run all을 실행하고 결과 기록을 다시 확인하세요. 설정·코드가 같으면 기존 run을 재사용합니다. 런타임을 재시작했다면 두 run을 다시 학습합니다. 결과가 달라지면 분석을 갱신하되 원래 예측은 유지하세요. READY는 제출 자료가 갖춰졌다는 뜻이며 분석의 정확성을 보장하지는 않습니다.

In [ ]:
#@title Export receipt / 결과 기록 PDF 생성
ai_use = '' #@param {type:"string"}
receipt = make_receipt(ai_use)